In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, current_timestamp, lit, from_json, explode

In [0]:
CATALOG       = "databricks_7405612194732360"
BRONZE_SCHEMA = "bronze_layer"
ARCHIVE_BASE  = f"/Volumes/{CATALOG}/raw_schema/raw/archive/"

RAW_PATHS = {
    "city_day":     f"/Volumes/{CATALOG}/raw_schema/raw/raw_city_day/",
    "city_hour":    f"/Volumes/{CATALOG}/raw_schema/raw/raw_city_hour/",
    "station_day":  f"/Volumes/{CATALOG}/raw_schema/raw/raw_station_day/",
    "station_hour": f"/Volumes/{CATALOG}/raw_schema/raw/raw_station_hour/",
}

DELTA_TABLES = {
    "city_day":     f"{CATALOG}.{BRONZE_SCHEMA}.bronze_city_day",
    "city_hour":    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_city_hour",
    "station_day":  f"{CATALOG}.{BRONZE_SCHEMA}.bronze_station_day",
    "station_hour": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_station_hour",
}

KEY_TYPE_MAP = {
    "city_day":     "city",
    "city_hour":    "city",
    "station_day":  "station",
    "station_hour": "station",
}

In [0]:
POLLUTANT_FIELDS = [
    StructField("PM2.5",      StringType(), True),
    StructField("PM10",       StringType(), True),
    StructField("NO",         StringType(), True),
    StructField("NO2",        StringType(), True),
    StructField("NOx",        StringType(), True),
    StructField("NH3",        StringType(), True),
    StructField("CO",         StringType(), True),
    StructField("SO2",        StringType(), True),
    StructField("O3",         StringType(), True),
    StructField("Benzene",    StringType(), True),
    StructField("Toluene",    StringType(), True),
    StructField("Xylene",     StringType(), True),
    StructField("AQI",        StringType(), True),
    StructField("AQI_Bucket", StringType(), True),
]

RAW_SCHEMAS = {
    "city_day":     StructType([StructField("City",      StringType(), True), StructField("Date",     StringType(), True)] + POLLUTANT_FIELDS),
    "city_hour":    StructType([StructField("City",      StringType(), True), StructField("Datetime", StringType(), True)] + POLLUTANT_FIELDS),
    "station_day":  StructType([StructField("StationId", StringType(), True), StructField("Date",     StringType(), True)] + POLLUTANT_FIELDS),
    "station_hour": StructType([StructField("StationId", StringType(), True), StructField("Datetime", StringType(), True)] + POLLUTANT_FIELDS),
}

In [0]:
def has_delta_data(path):
    try:
        names = [f.name for f in dbutils.fs.ls(path)]
        if "_delta_log/" not in names:
            return False
        return spark.read.format("delta").load(path).limit(1).count() > 0
    except Exception:
        return False

In [0]:
def ensure_delta_table_exists(table_full_name, key_type, schema):
    if spark.catalog.tableExists(table_full_name):
        return

    first_col  = "City" if key_type == "city" else "StationId"
    field_names = [f.name for f in schema.fields]
    time_col   = "Datetime" if "Datetime" in field_names else "Date"

    pollutant_ddl = "\n    ".join(
        [f"`{f.name}` STRING" for f in schema.fields
         if f.name not in (first_col, time_col, "AQI_Bucket")]
    )

    spark.sql(f"""
        CREATE TABLE {table_full_name} (
            `{first_col}`        STRING,
            `{time_col}`         STRING,
            {pollutant_ddl},
            `AQI_Bucket`         STRING,
            `ingestion_datetime` TIMESTAMP,
            `source`             STRING
        ) USING DELTA
    """)
    print(f"✅ Created table {table_full_name}")

In [0]:
# def process_and_append(key, raw_path, table_full_name, raw_schema):
#     df_raw = spark.read.format("delta").load(raw_path)

#     expected = set(f.name for f in raw_schema.fields)

#     distinct_keys_df = (
#         df_raw.withColumn("_m", from_json(col("raw_payload"), MapType(StringType(), StringType())))
#             .select(explode(col("_m")).alias("key", "value"))
#             .select("key")
#             .distinct()
#     )

#     actual = {row["key"] for row in distinct_keys_df.collect()}

#     extra, missing = actual - expected, expected - actual

#     if extra and missing:
#         msg = f"[{key}] SCHEMA MISMATCH — extra: {sorted(extra)} | missing: {sorted(missing)}"
#         print(f"   ❌ {msg}")
#         raise ValueError(msg)

#     if extra:
#         warn = f"[{key}] extra columns ignored: {sorted(extra)}"
#         print(f"   ⚠️  {warn}")
#         dbutils.fs.put(f"/Volumes/{CATALOG}/raw_schema/raw/schema_warnings/{key}.txt", warn, overwrite=True)

#     df_parsed = df_raw.withColumn("parsed", from_json(col("raw_payload"), raw_schema))
#     field_names = [f.name for f in raw_schema.fields]
#     for f in field_names:
#         df_parsed = df_parsed.withColumn(f, col(f"`parsed`.`{f}`"))
#     df_parsed = (df_parsed
#                  .withColumn("ingestion_datetime", current_timestamp())
#                  .withColumn("source", lit("unknown")))
#     df_final = df_parsed.select([col(f"`{c}`") for c in field_names + ["ingestion_datetime", "source"]])
#     df_final.write.format("delta").mode("append").saveAsTable(table_full_name)
#     print(f"   ✅ {df_final.count()} rows → {table_full_name}")

In [0]:
def process_and_append(key, raw_path, table_full_name, raw_schema):
    df_raw = spark.read.format("delta").load(raw_path)
    expected = set(f.name for f in raw_schema.fields)

    distinct_keys_df = (
        df_raw
        .withColumn("_m", from_json(col("raw_payload"), MapType(StringType(), StringType())))
        .select(explode(col("_m")).alias("key", "value"))
        .select("key")
        .distinct()
    )
    actual = {row["key"] for row in distinct_keys_df.collect()}
    extra   = actual - expected
    missing = expected - actual

    # ── SCHEMA MISMATCH: columns are both wrong AND missing → hard fail ───────
    if extra and missing:
        msg = f"[{key}] SCHEMA MISMATCH — extra: {sorted(extra)} | missing: {sorted(missing)}"
        print(f"   ❌ {msg}")
        raise ValueError(msg)   # This will fail the Databricks run → Airflow marks bronze FAILED

    # ── EXTRA COLUMNS ONLY: warn but continue ─────────────────────────────────
    if extra:
        warn = f"[{key}] extra columns ignored: {sorted(extra)}"
        print(f"   ⚠️  {warn}")
        dbutils.fs.put(
            f"/Volumes/{CATALOG}/raw_schema/raw/schema_warnings/{key}.txt",
            warn,
            overwrite=True
        )

    # ── Parse and write ───────────────────────────────────────────────────────
    df_parsed = df_raw.withColumn("parsed", from_json(col("raw_payload"), raw_schema))
    field_names = [f.name for f in raw_schema.fields]

    for f in field_names:
        df_parsed = df_parsed.withColumn(f, col(f"parsed.{f}"))

    df_parsed = (
        df_parsed
        .withColumn("ingestion_datetime", current_timestamp())
        .withColumn("source", lit("unknown"))
    )
    df_final = df_parsed.select([col(f"`{c}`") for c in field_names + ["ingestion_datetime", "source"]])
    df_final.write.format("delta").mode("append").saveAsTable(table_full_name)
    print(f"   ✅ {df_final.count()} rows → {table_full_name}")

In [0]:
def archive_and_clear(key, raw_path):
    archive_dest = ARCHIVE_BASE + key + "/"
    dbutils.fs.mkdirs(archive_dest)
    dbutils.fs.cp(raw_path, archive_dest, recurse=True)
    dbutils.fs.rm(raw_path, recurse=True)
    print(f"   📁 Archived [{key}] → {archive_dest}")

In [0]:
summary = []
failed_keys = []

for key, raw_path in RAW_PATHS.items():
    if not has_delta_data(raw_path):
        summary.append((key, "skipped — no data")); continue

    table_name, raw_schema, key_type = DELTA_TABLES[key], RAW_SCHEMAS[key], KEY_TYPE_MAP[key]

    try:    ensure_delta_table_exists(table_name, key_type, raw_schema)
    except Exception as e: summary.append((key, f"FAILED — table: {e}")); failed_keys.append(key); continue

    try:    process_and_append(key, raw_path, table_name, raw_schema)
    except ValueError as e: summary.append((key, f"FAILED — schema mismatch: {e}")); failed_keys.append(key); continue
    except Exception as e:  summary.append((key, f"FAILED — append: {e}")); failed_keys.append(key); continue

    try:    archive_and_clear(key, raw_path)
    except Exception as e:  summary.append((key, "PARTIAL — archived failed")); failed_keys.append(key); continue

    summary.append((key, "SUCCESS"))

print("\n── Summary ──")
for key, status in summary:
    print(f"  {key:<14} → {status}")

if failed_keys:
    raise Exception(f"Bronze layer failed for: {failed_keys}. See summary above.")